# 03 — LoRA vs Steering Vectors

**Objetivo (días 8–9):** implementar la misma modificación de comportamiento con steering (ya hecho) y con LoRA, y comparar resultados.

**Entregable:** tabla comparativa con resultados reales en calidad, coste y reversibilidad.

**Prerequisito:** haber completado los notebooks 01 y 02.

## Diccionario de conceptos

---

**Fine-tuning**  
Proceso de continuar el entrenamiento de un modelo preentrenado sobre un dataset específico para adaptar su comportamiento a una tarea o dominio concreto. Actualiza los pesos del modelo. Es más caro y permanente que el steering, pero puede producir cambios de comportamiento más profundos y generalizables.

---

**Full fine-tuning**  
Fine-tuning donde se actualizan todos los parámetros del modelo. Para GPT-2 (117M parámetros) es manejable; para modelos más grandes (7B, 70B) es prohibitivo en términos de memoria y tiempo de cómputo. Requiere almacenar gradientes y estados del optimizador para cada parámetro.

---

**PEFT (Parameter-Efficient Fine-Tuning)**  
Familia de técnicas que adaptan el comportamiento de un LLM actualizando solo una pequeña fracción de los parámetros. LoRA es el método más popular. El objetivo es conseguir resultados comparables al fine-tuning completo con una fracción del coste computacional y de memoria.

---

**LoRA (Low-Rank Adaptation)**  
Técnica PEFT que aproxima la actualización de pesos `ΔW` como el producto de dos matrices de bajo rango:

```
W' = W + ΔW ≈ W + B · A
```

donde `B ∈ ℝ^{d×r}` y `A ∈ ℝ^{r×k}` con rango `r ≪ min(d, k)`. Los pesos originales `W` se congelan; solo se entrenan `A` y `B`. La observación clave: empíricamente, los cambios de comportamiento necesarios en fine-tuning tienen rango intrínsecamente bajo, por lo que esta aproximación pierde poco. (Hu et al. 2021)

---

**Rango (rank, r)**  
El hiperparámetro central de LoRA. Controla cuántos parámetros se añaden al modelo. Con `r = 4` y GPT-2, se entrenan ~300K parámetros en lugar de 117M — una reducción del 99.7%. Un rango más alto captura cambios más complejos pero cuesta más memoria y cómputo.

---

**Adaptadores LoRA**  
Las matrices `A` y `B` que se añaden al modelo. Son módulos adicionales que se insertan en paralelo a las capas existentes (típicamente las proyecciones de atención: Q, K, V, O). Durante la inferencia, el output se calcula como `Wx + BAx`. Los adaptadores son pequeños archivos que se guardan por separado del modelo base.

---

**QLoRA**  
Extensión de LoRA que cuantiza el modelo base a 4 bits (formato NF4) para reducir su huella de memoria, y entrena los adaptadores LoRA en precisión completa. Permite hacer fine-tuning de modelos de 7B+ en una sola GPU de consumo. Para GPT-2 no es necesario, pero es el método estándar en modelos grandes. (Dettmers et al. 2023)

---

**Cuantización**  
Técnica de compresión que reduce la precisión numérica de los pesos del modelo (de float32 a int8, int4...). Reduce dramáticamente la memoria necesaria a costa de una ligera pérdida de precisión. NF4 (Normal Float 4-bit) es el formato de 4 bits optimizado para distribuciones de pesos de LLMs que usa QLoRA.

---

**Dataset de fine-tuning**  
El conjunto de ejemplos con los que se entrena el adaptador LoRA. Para modificación de comportamiento, cada ejemplo tiene un input y un output que demuestra el comportamiento deseado. La calidad importa más que la cantidad: 30–50 ejemplos bien curados suelen superar a 500 ejemplos ruidosos.

---

**Catastrophic forgetting**  
Fenómeno por el que el modelo pierde capacidades previas al ser entrenado en nuevos datos. Con fine-tuning completo y pocos datos puede ser severo. LoRA lo mitiga porque los pesos originales se mantienen congelados — el modelo base siempre está intacto y los adaptadores solo añaden el comportamiento nuevo.

---

**Generalización**  
Capacidad del modelo modificado de aplicar el nuevo comportamiento a prompts distintos a los del entrenamiento. Un método que memoriza el training set pero falla en prompts nuevos no generaliza. Evaluamos la generalización probando los mismos prompts en steering y LoRA y comparando cuál mantiene mejor el efecto fuera de los ejemplos vistos.

---

**Efectos secundarios**  
Cambios no deseados en el comportamiento del modelo producidos por la intervención. El steering agresivo puede hacer el texto incoherente en general; LoRA con pocos datos puede introducir artefactos o sesgos no deseados. Un buen método modifica lo que quieres cambiar y no toca lo demás.

---

**Reversibilidad**  
Steering: reversible al instante — basta con no registrar el hook. Los pesos no cambian. LoRA: no reversible en la misma sesión — hay que descargar los adaptadores y recargar el modelo base. Es una diferencia operacional importante en producción: con steering puedes alternar entre comportamientos en tiempo real.

---

**Coste de setup**  
Steering: segundos (una resta de vectores tras extraer activaciones de ~6 prompts). LoRA: minutos u horas dependiendo del tamaño del modelo, el dataset y el hardware. Para GPT-2 en Colab T4, un fine-tuning de 50 ejemplos a 3 épocas tarda típicamente 5–15 minutos.

## 1. Setup

Comparamos **dos formas de modificar el comportamiento de un LLM** sobre la misma tarea:

| Método | Qué modifica | Reversible | Coste de setup |
|--------|-------------|-----------|---------------|
| **Steering vectors** | Activaciones en inferencia | Sí — quitar el hook | Segundos |
| **LoRA** | Pesos del modelo (adaptadores) | No — hay que recargar el modelo base | Minutos / horas |

La comparativa es válida porque usamos el mismo modelo base (GPT-2), la misma tarea y los mismos prompts de evaluación. Cambiar cualquiera de estas tres variables haría la comparación injusta.

In [ ]:
!pip install transformer_lens peft transformers datasets -q

## 2. Tarea común

Modificación de comportamiento: **reducir certeza / aumentar escepticismo** en las respuestas del modelo. La misma tarea que el notebook 02, para poder comparar de forma justa.

Elegimos esta tarea porque:
1. Ya tenemos el steering vector calculado en el notebook 02 — podemos reutilizarlo directamente
2. Es una tarea de *comportamiento general* (no de conocimiento factual) — adecuada para GPT-2
3. El efecto es medible cualitativamente de forma simple: ¿aparece hedging en el output?

Usar la misma tarea es crítico: si cambiamos el objetivo, no podemos distinguir si las diferencias se deben al método o a la dificultad intrínseca de la tarea.

## 3. Steering (referencia desde notebook 02)

Reutilizamos el steering vector del notebook 02. El cálculo tarda segundos — lo más limpio es recalcularlo aquí. Alternativamente se puede serializar con `torch.save` y cargar con `torch.load`.

Necesitamos del notebook 02: el modelo cargado, la función `get_mean_activation`, los prompts contrastivos, y los valores de `LAYER` y `ALPHA` que dieron mejores resultados en la experimentación.

In [ ]:
# TODO: cargar modelo y recalcular steering vector del notebook 02
pass

## 4. LoRA con PEFT

**LoRA (Low-Rank Adaptation)** — Hu et al. 2021

La observación clave: cuando hacemos fine-tuning completo, los cambios en los pesos `ΔW` tienen rango intrínsecamente bajo. En lugar de actualizar `W` directamente, aproximamos:

```
W' = W + ΔW ≈ W + B · A
```

donde `B ∈ ℝ^{d×r}` y `A ∈ ℝ^{r×k}` con rango `r ≪ min(d, k)`.

Con `r = 4` en GPT-2, entrenamos ~300K parámetros en lugar de 117M — una reducción del 99.7%. Los pesos originales `W` se congelan durante el entrenamiento; solo se actualizan `A` y `B`.

**PEFT** de Hugging Face automatiza la inserción de los adaptadores LoRA. `get_peft_model()` envuelve el modelo y añade los módulos `A` y `B` donde se indique.

**Dataset — formato y tamaño:**  
Necesitamos pares `(input, output_escéptico)`. Ejemplos:
```
input:  "Scientists have discovered that the drug is effective."
output: "Scientists have suggested that the drug might be effective, though further studies are needed."

input:  "The new policy will reduce crime."
output: "The new policy could potentially reduce crime, though its effects remain uncertain."
```
Con 30–50 ejemplos y 3 épocas en T4 es suficiente para ver el efecto. La calidad importa más que la cantidad.

In [ ]:
# TODO: definir dataset de pares (input, output_escéptico)
pass

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoModelForCausalLM, AutoTokenizer

# TODO: cargar GPT-2 con PEFT y entrenar LoRA
pass

## 5. Comparativa

Dimensiones de comparación más allá de la calidad del output:

- **Generalización:** ¿funciona el método en prompts muy distintos a los usados para calcularlo/entrenarlo?
- **Efectos secundarios:** ¿el modelo pierde capacidad en otras tareas? ¿hay incoherencias?
- **Velocidad de setup:** steering es instantáneo; LoRA requiere entrenamiento
- **Memoria:** LoRA añade adaptadores; steering no añade parámetros
- **Reversibilidad:** steering se quita al instante; LoRA requiere recargar el modelo base

Documenta resultados reales — los teóricos ya están en la tabla de la sección 6.

In [ ]:
test_prompts = [
    "Scientists have discovered that",
    "The new policy will",
    "According to experts,",
    "The results of the study show",
    "People believe that",
]

# TODO: generar outputs con baseline, steering y LoRA
pass

## 6. Tabla de resultados

| Métrica | Baseline | Steering | LoRA |
|---------|----------|----------|------|
| Calidad / coherencia | | | |
| Efecto en comportamiento | | | |
| Generalización a prompts nuevos | | | |
| Efectos secundarios observados | — | | |
| Tiempo de setup | — | ~segundos | ~minutos |
| Parámetros modificados | — | 0 | ~300K (r=4) |
| Memoria adicional | — | 0 | adaptadores |
| Reversible al instante | — | Sí | No |